# Unit 7 Practical: Complete System Simulation (Student)
## Cart-Pendulum with Motor Drive

### Objectives
- Simulate a complete multi-body dynamic system
- Compare numerical integration methods
- Implement external forcing and control
- Validate results through energy analysis
- Produce publication-quality visualizations

### System Description

A motor-driven cart on a horizontal track with an attached pendulum:

```
                   ●  m_p (pendulum mass)
                  /|
                /  |
              /    | L (pendulum length)
            /      |
          / θ      |
  ━━━━━━━━━━━━━━━━
  |   Cart m_c   |  ←── F_motor(t)
  ━━━━━━━━━━━━━━━━
  ═════════════════  (frictionless track)
```

**Key Features**:
- **Cart**: Mass $m_c$, position $x$, driven by motor force $F_{motor}(t)$
- **Pendulum**: Mass $m_p$, length $L$, angle $\theta$ from vertical
- **Forcing**: Time-varying motor force
- **Constraints**: Cart moves only horizontally, pendulum pivots at cart center

**Applications**:
- Inverted pendulum control
- Crane anti-sway systems
- Robot balance control
- Seismic vibration testing

### Learning Outcomes
1. Formulate multi-body EOM using Lagrangian mechanics
2. Implement system in simulation-ready form
3. Compare ODE solver performance
4. Analyze dynamic response
5. Validate through energy conservation

**Duration**: ~150 minutes (2.5 hours)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time
from matplotlib.patches import Rectangle, Circle, FancyArrow
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Configure plotting
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10

print("="*70)
print("UNIT 7 PRACTICAL: Cart-Pendulum System Simulation")
print("="*70)

## Part 1: System Formulation

### Lagrangian Approach

#### Generalized Coordinates
- $q_1 = x$ (cart position)
- $q_2 = \theta$ (pendulum angle from vertical)

#### Kinetic Energy
**Cart**: $T_c = \frac{1}{2}m_c\dot{x}^2$

**Pendulum** (position: $(x + L\sin\theta, L\cos\theta)$):
$$T_p = \frac{1}{2}m_p\left[(\dot{x} + L\dot{\theta}\cos\theta)^2 + (L\dot{\theta}\sin\theta)^2\right]$$

**Total**:
$$T = \frac{1}{2}(m_c + m_p)\dot{x}^2 + \frac{1}{2}m_pL^2\dot{\theta}^2 + m_pL\dot{x}\dot{\theta}\cos\theta$$

#### Potential Energy
$$V = -m_pgL\cos\theta$$ (taking cart level as zero)

#### Lagrangian
$$L = T - V$$

#### Equations of Motion
Using Euler-Lagrange equations with generalized force $F_{motor}$ on cart:

$$(m_c + m_p)\ddot{x} + m_pL\ddot{\theta}\cos\theta - m_pL\dot{\theta}^2\sin\theta = F_{motor}$$

$$m_pL^2\ddot{\theta} + m_pL\ddot{x}\cos\theta - m_pgL\sin\theta = 0$$

#### Convert to First-Order System
State vector: $\mathbf{y} = [x, \theta, \dot{x}, \dot{\theta}]^T$

$$\frac{d\mathbf{y}}{dt} = \begin{bmatrix} \dot{x} \\ \dot{\theta} \\ \ddot{x} \\ \ddot{\theta} \end{bmatrix}$$

Solve coupled equations for $\ddot{x}$ and $\ddot{\theta}$:

$$\ddot{x} = \frac{F_{motor} + m_pL\dot{\theta}^2\sin\theta + m_pg\sin\theta\cos\theta}{m_c + m_p\sin^2\theta}$$

$$\ddot{\theta} = \frac{-F_{motor}\cos\theta - m_pL\dot{\theta}^2\sin\theta\cos\theta + (m_c+m_p)g\sin\theta}{L(m_c + m_p\sin^2\theta)}$$

In [ ]:
# Part 1: System Parameters and Implementation
print("\n" + "="*70)
print("PART 1: System Formulation and Implementation")
print("="*70)

# Physical parameters
m_c = 2.0    # kg (cart mass)
m_p = 0.5    # kg (pendulum mass)
L = 1.0      # m (pendulum length)
g = 9.81     # m/s² (gravity)

print(f"\nSystem parameters:")
print(f"Cart mass: m_c = {m_c} kg")
print(f"Pendulum mass: m_p = {m_p} kg")
print(f"Pendulum length: L = {L} m")
print(f"Gravity: g = {g} m/s²")

# Motor forcing function - sinusoidal with envelope
def motor_force(t):
    """
    Time-varying motor force with:
    - Sinusoidal component at frequency f
    - Gaussian envelope centered at t_center
    """
    f = 0.5  # Hz
    t_center = 5.0  # s
    sigma = 2.0  # s
    amplitude = 5.0  # N
    
    envelope = np.exp(-(t - t_center)**2 / (2*sigma**2))
    return amplitude * envelope * np.sin(2*np.pi*f*t)

# System equations of motion
def cart_pendulum_ode(t, y):
    """
    Cart-pendulum equations of motion
    State: y = [x, θ, ẋ, θ̇]
    Returns: dy/dt = [ẋ, θ̇, ẍ, θ̈]
    """
    x, theta, x_dot, theta_dot = y
    
    # Motor force at current time
    F = motor_force(t)
    
    # Common denominator
    denom = m_c + m_p * np.sin(theta)**2
    
    # Accelerations from coupled equations
    x_ddot = (F + m_p*L*theta_dot**2*np.sin(theta) 
              + m_p*g*np.sin(theta)*np.cos(theta)) / denom
    
    theta_ddot = (-F*np.cos(theta) - m_p*L*theta_dot**2*np.sin(theta)*np.cos(theta)
                  + (m_c + m_p)*g*np.sin(theta)) / (L * denom)
    
    return [x_dot, theta_dot, x_ddot, theta_ddot]

# Initial conditions
x0 = 0.0           # Cart starts at origin
theta0 = np.pi/6   # Pendulum at 30° from vertical
x_dot0 = 0.0       # Cart at rest
theta_dot0 = 0.0   # Pendulum at rest

y0 = [x0, theta0, x_dot0, theta_dot0]

print(f"\nInitial conditions:")
print(f"Cart position: x₀ = {x0} m")
print(f"Pendulum angle: θ₀ = {np.degrees(theta0):.1f}° from vertical")
print(f"Cart velocity: ẋ₀ = {x_dot0} m/s")
print(f"Pendulum angular velocity: θ̇₀ = {theta_dot0} rad/s")

# Test motor forcing
t_test = np.linspace(0, 10, 500)
F_test = [motor_force(t) for t in t_test]

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(t_test, F_test, 'b-', linewidth=2)
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Motor Force (N)')
ax.set_title('Motor Forcing Function: F(t) = A·exp(-(t-5)²/8)·sin(2πft)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/Teacher/figs/u7_p1_forcing.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSystem formulation complete")
print(f"Motor forcing function defined")
print("="*70)
# TODO: Use these parameters in your solutions below


## Part 2: Numerical Method Comparison

We'll compare four integration methods:

| Method | Order | Characteristics |
|--------|-------|-----------------|
| **RK4 (manual)** | 4 | Classical implementation |
| **RK23** | 3 | scipy adaptive, lower accuracy |
| **RK45** | 5 | scipy adaptive, general purpose |
| **DOP853** | 8 | scipy adaptive, high accuracy |

**Evaluation Criteria**:
1. **Accuracy**: Compare with high-precision reference
2. **Computational cost**: Function evaluations
3. **Energy conservation**: Physical validation
4. **Execution time**: Wall-clock performance

In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

## Part 3: System Visualization

Create a comprehensive visualization showing:
1. **System state**: Cart and pendulum configuration
2. **Trajectory**: Pendulum mass path
3. **Time histories**: Positions and velocities
4. **Energy flow**: Tracking energy transfer

This provides physical insight into system behavior.

In [ ]:
# TODO: Create visualization# Hint: Use matplotlib to plot your results## Suggested structure:# 1. Create figure and axes# 2. Plot calculated results# 3. Add labels and formatting# 4. Display the plot# Your code here:

## Summary and Key Takeaways

### System Analysis

**Cart-Pendulum Dynamics**:
- Coupled 2-DOF system with motor forcing
- Lagrangian formulation yields coupled ODEs
- Energy flows between cart KE, pendulum KE, and PE
- Motor work input tracked and validated

### Numerical Methods

**Performance Rankings**:

1. **RK45** (Winner for most applications)
   - Good accuracy with reasonable cost
   - Adaptive step size
   - ~1000-2000 function evaluations
   - Excellent energy conservation

2. **DOP853** (High accuracy)
   - Best accuracy
   - Higher computational cost
   - ~2000-4000 function evaluations
   - Use when precision is critical

3. **RK23** (Quick solutions)
   - Lower accuracy
   - Fastest execution
   - ~500-1000 function evaluations
   - Good for rough estimates

4. **RK4 Manual** (Educational/special cases)
   - Fixed step size (predictable cost)
   - Good accuracy with small h
   - 4× steps function evaluations
   - Useful when adaptive stepping unwanted

### Validation Techniques

1. **Energy Conservation**
   - Track $E = KE_{cart} + KE_{pendulum} + PE$
   - Should equal $E_0 + W_{motor}$
   - Errors indicate numerical drift

2. **Method Comparison**
   - Use high-accuracy method as reference
   - Compare positions/velocities
   - Quantify accuracy vs cost trade-off

3. **Physical Reasonableness**
   - Visualize motion
   - Check for unphysical behavior
   - Verify constraint satisfaction

### Best Practices for Simulation Projects

1. **Formulation**
   - Use Lagrangian for complex systems
   - Identify all generalized coordinates
   - Convert to first-order ODEs carefully

2. **Implementation**
   - Write clean, modular ODE function
   - Test with simple cases first
   - Validate against known solutions

3. **Method Selection**
   - Start with RK45 (good default)
   - Use DOP853 for high accuracy
   - Consider stiff solvers if needed

4. **Validation**
   - Always check conservation laws
   - Compare methods
   - Visualize results
   - Verify physical correctness

5. **Documentation**
   - Clear variable names
   - Document equations
   - Explain method choices
   - Present results professionally

### Applications in Engineering

This cart-pendulum simulation approach applies to:

- **Robotics**: Manipulator dynamics, humanoid balance
- **Control Systems**: Inverted pendulum control, stabilization
- **Mechanical Design**: Crane systems, load handling
- **Automotive**: Suspension dynamics, rollover analysis
- **Aerospace**: Satellite attitude control, launch vehicle dynamics

### Course Completion

Congratulations! You've completed a comprehensive journey through:

1. **Kinematics**: Position, velocity, acceleration in multiple frames
2. **Lagrangian Mechanics**: Energy-based dynamics formulation
3. **Rigid Body Kinematics**: Rotation, angular velocity, Euler angles
4. **Particle/Rigid Body Kinetics**: Forces, torques, equations of motion
5. **Vibrations**: SDOF/MDOF systems, modal analysis
6. **Work & Energy**: Energy methods, momentum methods
7. **Numerical Simulation**: ODE solvers, multi-body dynamics

You now have the tools to:
- Formulate equations of motion for complex systems
- Simulate dynamic behavior numerically
- Analyze and validate results
- Apply to real engineering problems

**Next steps**: Apply these methods to your own projects, systems, and research!